In [18]:
!git clone https://github.com/mohamedrafat9/ticketllm.git
%cd ticketllm

Cloning into 'ticketllm'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 76 (delta 24), reused 66 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 16.74 KiB | 5.58 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/ticketllm/ticketllm/ticketllm/ticketllm


In [19]:
from src.inference.load_model import load_model
from src.inference.generate import generate_text as g

In [5]:
tokenizer, model = load_model()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

...

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [6]:
# messages = [
#     {
#         "role": "system",
#         "content": (
#             "You are a english arabic translator"
#             "i will give you the pharse in english , you returned in arabic "
#             'Return only valid JSON using this schema: '
#             # '{"sentiment": "Positive|Negative|Neutral"}'
#         )
#     },
#     {
#         "role": "user",
#         "content": "1.This movie was great"
#     },
#     {
#         "role": "assistant",
#         "content": '{"1": "الفيلم كان رائعا"}'
#     },
#     {
#         "role": "user",
#         "content": "2.I hate this movie"
#     },
#     {
#         "role": "assistant",
#         "content": '{"2": "انا اكره هذا الفيلم"}'
#     },
#     {
#         "role": "user",
#         "content": "3.This movie was okay"
#     },
#     {
#         "role": "assistant",
#         "content": '{"3": "الفيلم كان كويس"}'
#     },
#     {
#         "role": "user",
#         "content": "4.I don't know what this movie was about"

#     }
# ]

# output = g(
#     model,
#     tokenizer,
#     messages,
#     temperature=1,
#     top_k=50,
#     max_new_tokens=300
# )

# print(output)

In [7]:
# !pip install pydantic

In [8]:
import json
import re
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

class TicketAnalysis(BaseModel):
    category: Literal["Bug", "Feature Request", "Billing", "Technical Support"] = Field(
        ..., 
        description="classification of the ticket based on its content"
    )
    priority: Literal["Low", "Medium", "High", "Critical"] = Field(
        ..., 
        description="priority of the ticket based on its impact"
    )
    sentiment: Literal["Frustrated", "Neutral", "Satisfied"] = Field(
        ..., 
        description="customer's sentiment and tone in describing the issue"
    )
    summary: str = Field(
        ..., 
        min_length=5, 
        max_length=120, 
        description="direct summary of the issue or concern in a single line"
    )
    action_required: bool = Field(
        ..., 
        description="indicates whether the ticket requires immediate action or escalation"
    )

system_prompt = f"""You are an automated IT Support Classifier for an enterprise Ticketing System.
Analyze the user's support ticket and extract structured details.

You MUST return ONLY a valid JSON object matching this schema:
{json.dumps(TicketAnalysis.model_json_schema(), indent=2)}

Do not wrap output in markdown syntax (e.g. no ```json). Return raw JSON only."""

messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user", 
        "content": "Subject: App crashes on launch\nBody: Every time I open the app today after the update, it closes immediately on my iPhone. Fix this ASAP!"
    },
    {
        "role": "assistant", 
        "content": json.dumps({
            "category": "Bug",
            "priority": "High",
            "sentiment": "Frustrated",
            "summary": "App crashes immediately upon launch after update on iOS",
            "action_required": True
        })
    },
    {
        "role": "user", 
        "content": "Subject: Double charge on invoice #1042\nBody: Hi, I checked my bank statement and saw that my subscription was billed twice this morning. Could someone check this and refund the extra amount?"
    }
]


raw_output = g(
    model,
    tokenizer,
    messages,
    temperature=0.1,
    top_k=50,
    max_new_tokens=250
)


def parse_ticket(text: str) -> TicketAnalysis | None:
    clean_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    try:
        ticket = TicketAnalysis.model_validate_json(clean_text)
        return ticket
    except ValidationError as e:
        print("Validation Error:\n", e)
        return None


parsed_ticket = parse_ticket(raw_output)

if parsed_ticket:
    print("Extracted Ticket Data:\n", parsed_ticket.model_dump_json(indent=2))
    print("\nAccessed Attributes:")
    print("Category:", parsed_ticket.category)
    print("Priority:", parsed_ticket.priority)
    print("Requires Escalation:", parsed_ticket.action_required)

OutOfMemoryError: CUDA out of memory. Tried to allocate 194.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 179.81 MiB is free. Including non-PyTorch memory, this process has 14.38 GiB memory in use. Of the allocated memory 13.73 GiB is allocated by PyTorch, and 547.11 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [9]:
print(model.dtype)

print(
    sum(p.numel() for p in model.parameters()) / 1e9,
    "B parameters"
)

torch.bfloat16
3.83602176 B parameters


In [10]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_summary())

Tesla T4
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 1            |        cudaMalloc retries: 1         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  14054 MiB |  14066 MiB |  29482 MiB |  15427 MiB |
|       from large pool |  13927 MiB |  13939 MiB |  28982 MiB |  15055 MiB |
|       from small pool |    127 MiB |    128 MiB |    500 MiB |    372 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  14054 MiB |  14066 MiB |  29482 MiB |  15427 MiB |
|       from large pool |  13927 MiB |  13939 MiB |  28

In [20]:
import importlib
import src.inference.generate

importlib.reload(src.inference.generate)

from src.inference.generate import generate_text

In [22]:
import inspect
from src.inference.generate import generate_text

print(inspect.getsource(generate_text))

def generate_text(model, tokenizer, prompt, max_length=50, temperature=1.0,top_p=None, top_k=None,max_new_tokens=50):
    device = get_model_device(model)
    inputs = tokenize(prompt, tokenizer, device)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']
    for _ in range(max_new_tokens):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        next_token_logits = outputs.logits[:, -1, :]
        next_token_logits = apply_temperature(next_token_logits, temperature)
        next_token_logits = apply_top_k(next_token_logits, top_k) if top_k is not None else next_token_logits
        # next_token_logits = apply_top_p(next_token_logits, top_p) if top_p
        next_token = select_next_token(next_token_logits, temperature)
        if(tokenizer.eos_token_id is not None
            and next_token.item() == tokenizer.eos_token_id):
            break
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        attention_mask =